# 12a · Synchronization summary

Export the synchronization workbook and data-availability CSV.


In [1]:
import sys
from pathlib import Path
from IPython.display import display

PROJECT_ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
                    if (p / 'helpers/project_config.py').is_file())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from helpers.project_config import load_project_config
from helpers.sync_quality import report

CONFIG = load_project_config(PROJECT_ROOT / 'config.json')
PROJECT_NAME = CONFIG.project
PREPROCESSED_ROOT = CONFIG.path('paths', 'preprocessed_data_root')
COVERAGE_CACHE = PROJECT_ROOT / '.cache' / 'sync_quality' / 'coverage_modules.json'
OUTPUT_FILE = CONFIG.path('files', 'sq_summary_xlsx')
OVERWRITE = True
FORCE_REBUILD_COVERAGE = False
QC_TABLE_PATHS = {key: CONFIG.path('files', key)
                  for key in CONFIG.values['files'] if key.endswith('_alignment_csv')}
QC_TABLE_PATHS, QC_TABLES = report.load_qc_tables(QC_TABLE_PATHS)
print('Publication data:', PREPROCESSED_ROOT)
print('Output:', OUTPUT_FILE)


Publication data: E:\test\DatasetForPublic\preprocessed_data
Output: E:\ImBrAid\code\ImBrAid\pipeline\Sync_and_Quality\docs\sync_quality_summary.xlsx


In [2]:
COVERAGE, COVERAGE_INFO = report.load_or_build_coverage(
    PREPROCESSED_ROOT, QC_TABLES, scope='modules', project=PROJECT_NAME,
    force=FORCE_REBUILD_COVERAGE, cache_path=COVERAGE_CACHE)
print('Coverage:', COVERAGE_INFO['status'])
USER_SUMMARY = report.build_user_summary(
    PREPROCESSED_ROOT, QC_TABLES, COVERAGE, project=PROJECT_NAME)
display(USER_SUMMARY.head(12))

Coverage: cache hit


,Recording ID,Data type,Data available,Time alignment,Use note
0,A7G5SM_VX,Eye tracking,XDF + TSV,LSL,20.2% of driving-module samples outside timing...
1,A7G5SM_VX,Eyeball pupil,XDF + CSV,LSL,19.8% of driving-module samples outside timing...
2,A7G5SM_VX,Driver landmarks,XDF + CSV,LSL,20.3% of driving-module samples outside timing...
3,A7G5SM_VX,Ear EEG,Not provided,Not applicable,
4,ABVLGQ_VX,Eye tracking,XDF + TSV,LSL,
5,ABVLGQ_VX,Eyeball pupil,XDF + CSV,LSL,
6,ABVLGQ_VX,Driver landmarks,XDF + CSV,LSL,
7,ABVLGQ_VX,Ear EEG,XDF + CSV,LSL,
8,B9NYQ8_VX,Eye tracking,XDF + TSV,LSL,
9,B9NYQ8_VX,Eyeball pupil,XDF + CSV,LSL,


In [3]:
from pathlib import Path
import pandas as pd
from openpyxl import Workbook

columns = ['Recording ID', 'Data type', 'Data available', 'Time alignment', 'Use note']
if list(USER_SUMMARY.columns) != columns:
    raise ValueError('USER_SUMMARY must contain the five documented columns in order.')
if USER_SUMMARY.empty:
    raise ValueError('No recording rows are available for the user summary.')

output_file = Path(OUTPUT_FILE)
if output_file.suffix.lower() != '.xlsx':
    raise ValueError('OUTPUT_FILE must have an .xlsx extension.')
if output_file.exists() and not OVERWRITE:
    raise FileExistsError(f'Summary already exists: {output_file}')

workbook = Workbook()
sheet = workbook.active
sheet.title = 'Sync summary'
sheet['A2'] = 'ImBrAid synchronization summary'
notes = [
    'Data available lists processed formats found in the configured publication directory.',
    'LSL is the shared clock used in XDF. Timing of CSV/TSV-only data is not assessed here.',
    'Outside timing support means beyond the saved mapping interval; percentages use XDF samples within the 10 driving modules only.',
    'Review flags come from saved synchronization QC; they do not assess EEG or eye-tracking signal quality.',
]
for row_number, text in enumerate(notes, start=3):
    sheet.cell(row_number, 1, text)
for column_number, name in enumerate(columns, start=1):
    sheet.cell(8, column_number, name)
for row_number, values in enumerate(USER_SUMMARY.itertuples(index=False, name=None), start=9):
    for column_number, value in enumerate(values, start=1):
        cell = sheet.cell(row_number, column_number, '' if pd.isna(value) else str(value))
        cell.data_type = 's'

output_file.parent.mkdir(parents=True, exist_ok=True)
workbook.save(output_file)
print(f'Saved {len(USER_SUMMARY)} recording/data-type rows to {output_file}')


Saved 172 recording/data-type rows to E:\ImBrAid\code\ImBrAid\pipeline\Sync_and_Quality\docs\sync_quality_summary.xlsx


In [4]:
AVAILABILITY_FILE = CONFIG.path('files', 'data_avalibility_csv')
if AVAILABILITY_FILE.exists() and not OVERWRITE:
    raise FileExistsError(f'Availability table already exists: {AVAILABILITY_FILE}')
DATA_AVAILABILITY = report.build_data_availability(PREPROCESSED_ROOT, QC_TABLES['et_lsl_alignment_csv'])
AVAILABILITY_FILE.parent.mkdir(parents=True, exist_ok=True)
DATA_AVAILABILITY.replace({True: 'true', False: 'false'}).to_csv(AVAILABILITY_FILE, index=False)
print(f'Saved {len(DATA_AVAILABILITY)} recording rows to {AVAILABILITY_FILE}')


Saved 43 recording rows to E:\ImBrAid\code\ImBrAid\pipeline\Sync_and_Quality\docs\data_availability.csv
